## BRCA prediction with the UNI2 pretrained model

Score Xenium BRCA rep1 and rep2 with the pan-cancer UNI2 checkpoint chosen from the A0–A3 ablation:

`data/PanCancer/result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/cross_dataset_cv/A2_adjacent_spatial_k8mean/best_mlp_gpu.pt`

A2 is adjacent loss plus spatial k=8 mean and no gate. This is the encoder-comparison weight (paired_v2). The BRCA-only UNI training notebook stays in `BRCA_train_validate_cv_UNIlabel_all.ipynb`.

- Embeddings are the UNI2 class token, `Cases/{rep}/project_all_UNI2/ImgEmbeddings_all/sc_pth_14_14` (1536-d). `h5ad.X` is still UNI.
- Spatial context stays on. The checkpoint stores k=8 mean and no gate.
- `--deployment` applies this one file to both replicates. The scores are not cross-section OOF.
- `best_mlp_gpu.pt` is fold 1 of `result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/fold_plan.csv`. The score cell marks that fold. Those sections were not used to fit this file.
- Labels are the pan-cancer ontology. ROC tables are GT-matched cells.
- All-nuclei StarDist maps use `ImgEmbeddings_all_stardist/sc_pth_14_14`. The store is `embeddings.h5ad`. Extract it if that file is missing, then re-run the predict cell; a finished matched-cell section still writes the missing all-nuclei h5ad.

Env: `conda activate SeededNTM`. Run the cells in order. `RUN_A2_PRED = True` with `--skip-existing` leaves a finished section in place. Set `SKIP_A2_FIGURES = False` to draw the matched-cell panel; a section whose panel PNG is missing is recomputed.


In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ.setdefault("NCRT_CUDA_DEVICE", "0")

import sys
from pathlib import Path

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
for _d in (REPO / "code/Hist2Pheno_pkg", REPO / "code/PanCancer"):
    if str(_d) not in sys.path:
        sys.path.insert(0, str(_d))

from runtime_env import configure_notebook_runtime
_rt_env = configure_notebook_runtime()
print(f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')} (physical GPU {_rt_env['physical_gpu']})")


In [ ]:
import subprocess

BEST_FOLD = 1  # paired_v2 A2 best_mlp
A2_CKPT = (
    REPO
    / "data/PanCancer/result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42"
    / "cross_dataset_cv/A2_adjacent_spatial_k8mean/best_mlp_gpu.pt"
)
A2_PRED = REPO / "data/PanCancer/pred_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42_deployment"
A2_ROC = A2_PRED / "stardist_roc"
RUN_A2_PRED = True
SKIP_A2_FIGURES = True

if not A2_CKPT.is_file():
    raise FileNotFoundError(A2_CKPT)

cmd = [
    sys.executable,
    "-u",
    str(REPO / "code/PanCancer/PanCancer_predict_visualize.py"),
    "--cuda-device",
    os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0],
    "--embedder",
    "uni2",
    "--deployment",
    "--spatial-k",
    "8",
    "--datasets",
    "BRCA",
    "--skip-existing",
    "--checkpoint",
    str(A2_CKPT),
    "--pred-root",
    str(A2_PRED),
    "--data-root",
    str(REPO / "data/PanCancer"),
]
if SKIP_A2_FIGURES:
    cmd.append("--skip-figures")
print(" ".join(cmd))
if RUN_A2_PRED:
    subprocess.run(cmd, check=True, cwd=str(REPO))
else:
    print("RUN_A2_PRED=False; reading ROC tables under", A2_ROC)


In [ ]:
import numpy as np
import pandas as pd
from uni_label_cv_helpers import (
    macro_auc_ovr,
    plot_pooled_stardist_tier_rocs,
    stardist_auroc_tier_spec,
    symlink_stardist_sample_dirs,
)

DATASET = "BRCA"
FOLD_PLAN = REPO / "data/PanCancer/result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/fold_plan.csv"
plan = pd.read_csv(FOLD_PLAN)
plan = plan.loc[plan["dataset"].astype(str) == DATASET].copy()
if plan.empty:
    raise RuntimeError(f"No {DATASET} rows in {FOLD_PLAN}")
plan = plan.sort_values(["fold", "sample"])
SAMPLES = {
    str(row.sample): f"{DATASET}__{row.sample}"
    for row in plan.itertuples(index=False)
}
HELD_OUT = set(plan.loc[plan["fold"].astype(int) == BEST_FOLD, "sample"].astype(str))
TIERS = ("l2", "l12", "l1")


def section_macro(sample_dir: Path, tier: str):
    spec = stardist_auroc_tier_spec(tier)
    df = pd.read_csv(sample_dir / spec["csv_name"])
    prob_cols = [c for c in df.columns if c.startswith("prob_")]
    y = df["ground_truth_idx"].to_numpy()
    score = df[prob_cols].to_numpy(dtype=np.float64)
    return macro_auc_ovr(y, score, score.shape[1]), int(len(df))


rows = []
for name, dirname in SAMPLES.items():
    sample_dir = A2_ROC / dirname
    if not sample_dir.is_dir():
        raise FileNotFoundError(sample_dir)
    for tier in TIERS:
        auc, n = section_macro(sample_dir, tier)
        rows.append({
            "sample": name,
            "held_out": name in HELD_OUT,
            "tier": tier.upper(),
            "n_cells": n,
            "macro_auroc": auc,
        })

per_sample = pd.DataFrame(rows)
print(
    f"Per-section macro AUROC on GT-matched cells. "
    f"{len(HELD_OUT)} / {plan['sample'].nunique()} sections are held out of best_mlp (fold {BEST_FOLD})."
)
wide = per_sample.pivot(index="sample", columns="tier", values="macro_auroc")
wide = wide.reindex(columns=["L2", "L12", "L1"])
wide.insert(0, "held_out", wide.index.isin(HELD_OUT))
display(wide)

cohort_roc = A2_PRED / "brca_roc"
held_names = [f"{DATASET}__{name}" for name in SAMPLES if name in HELD_OUT]
held_roc = A2_PRED / "brca_heldout_roc"
symlink_stardist_sample_dirs(A2_ROC, list(SAMPLES.values()), cohort_roc)
symlink_stardist_sample_dirs(A2_ROC, held_names, held_roc)


def pool(root, out, label):
    out.mkdir(parents=True, exist_ok=True)
    results = plot_pooled_stardist_tier_rocs(
        root,
        out,
        tiers=TIERS,
        layout="pooled_stardist",
        pan_organ="pancancer",
        figsize=(2.0, 2.0),
        show_barplot=False,
        save_per_class_csv=True,
        show=True,
    )
    print(label)
    for tier, rec in results.items():
        info = rec.get("roc_info") or {}
        print(f"  pooled {tier.upper()} macro AUROC = {info.get('macro_auc', float('nan')):.4f}")
    return results


results_all = pool(
    cohort_roc,
    A2_PRED / "brca_pooled_roc",
    "All BRCA sections. This pool includes sections the weight trained on.",
)
results_held = pool(
    held_roc,
    A2_PRED / "brca_pooled_roc_heldout",
    f"Fold {BEST_FOLD} only. These sections were held out of best_mlp.",
)


## All StarDist nuclei

Every StarDist nucleus in this cohort, with no matched-cell AUROC. It uses the same A2 checkpoint. It does not retrain a BRCA-only model.

GT tokens cannot stand in for StarDist centroids. Extract first (patch images off):

```bash
cd /home/lingyu/ssd2/Python/Hist2Pheno
SAVE_PATCH_IMAGES=0 bash code/Hist2Pheno_pkg/run_uni2_patch14.sh brca stardist 0
```

Writes `Cases/{rep}/project_all_UNI2/ImgEmbeddings_all_stardist/sc_pth_14_14/embeddings.h5ad`. Then re-run the predict cell. With `--skip-existing`, a section is predicted again only when its all-nuclei h5ad is missing.


In [ ]:
import json
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from uni_label_cv_helpers import (
    plot_stardist_label_spatial_heads,
    plot_stardist_label_spatial_overview,
)

HEADS = ("l2", "l12", "l1")
OBS_COL = {"l2": "pred_CT", "l12": "pred_sublineage", "l1": "pred_lineage"}
CLASS_KEY = {"l2": "L2", "l12": "L12", "l1": "L1"}
classes = json.loads((A2_PRED / "ontology_heads.json").read_text())
loaded = {}
missing = []
for sample in ("rep1", "rep2"):
    token_dir = (
        REPO / "data/Xemium/BRCA/Cases" / sample
        / "project_all_UNI2/ImgEmbeddings_all_stardist/sc_pth_14_14"
    )
    path = A2_PRED / "stardist_all" / "BRCA" / sample / f"{sample}_all_stardist_pred.h5ad"
    if not (token_dir / "embeddings.h5ad").is_file() and not (token_dir / "embeddings.h5").is_file():
        missing.append(f"{sample}: no UNI2 StarDist embeddings.h5ad at {token_dir}")
        continue
    if not path.is_file():
        missing.append(f"{sample}: tokens exist; re-run the predict cell to write {path.name}")
        continue
    adata = ad.read_h5ad(path)
    coords = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)[:, :2]
    tiers = {}
    for head in HEADS:
        names = [str(x) for x in classes[CLASS_KEY[head]]]
        index = {name: i for i, name in enumerate(names)}
        pred = np.fromiter(
            (index[str(v)] for v in adata.obs[OBS_COL[head]].astype(str)),
            dtype=np.int64,
            count=adata.n_obs,
        )
        tiers[head] = {"class_names": names, "pred_encoded": pred}
    loaded[sample] = {"path": path, "coords": coords, "n_obs": int(adata.n_obs), "tiers": tiers}
    saved = plot_stardist_label_spatial_heads(
        loaded[sample],
        sample=sample,
        heads=HEADS,
        pan_organ="pancancer",
        show=False,
        title_prefix=f"BRCA {sample} A2",
    )
    for head in HEADS:
        display(Image(filename=str(saved[head])))

if missing:
    print("All-nuclei maps are not ready.")
    for line in missing:
        print(" ", line)
    print("Extract, then re-run the predict cell:")
    print("  SAVE_PATCH_IMAGES=0 bash code/Hist2Pheno_pkg/run_uni2_patch14.sh brca stardist 0")
elif loaded:
    overview = A2_PRED / "stardist_all" / "BRCA" / "stardist_all_pred_overview.jpg"
    fig = plot_stardist_label_spatial_overview(
        loaded,
        heads=HEADS,
        pan_organ="pancancer",
        save_path=overview,
        show=False,
        suptitle="BRCA all StarDist nuclei — UNI2 A2",
    )
    if fig is not None:
        plt.close(fig)
    display(Image(filename=str(overview)))
    print(f"Overview → {overview}")
